In [ ]:
import os
from time import perf_counter as now

import cv2
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

MMSEG_ROOT = '/mmsegmentation'
os.chdir(MMSEG_ROOT)

from mmseg.apis import init_model
from mmseg.utils import register_all_modules

# ============================================================
# PATHS / CONFIGURATION
# ============================================================
CONFIG_PATH = '/mmsegmentation/zmax_configs/for_test_hasta_26_3/multitask_test_sin_trigger_clean.py'
CKPT_PATH   = '/mmsegmentation/work_dirs/multitask_val_fix_wloss/best_cls_acc_cls_top1_iter_25600.pth'

INPUT_VIDEO = '/0_secuencia_paravideo/1a_secuencia2.mp4'
DEVICE = 'cuda:0' 

# Adaptive Clockwork
# Policy: if the mode changes (STRAIGHT <-> CURVE), the next frame is forced as FIRE.
K_STRAIGHT = 100
K_CURVE = 30
TRIGGER_IDX_STRAIGHT = 0
TRIGGER_IDX_CURVE = 1
TRIGGER_NAMES = ['STRAIGHT', 'CURVE']

# Multitask classification indices:
CLS_IDX_LEFT = 0
CLS_IDX_STRAIGHT = 1
CLS_IDX_RIGHT = 2

# GPU-only benchmark
WARMUP_FRAMES = 60
MAX_FRAMES = None            # None = uses the entire video
GPU_SAMPLE_EVERY = 1         # 1 = measures all frames
GPU_SAMPLE_OFFSET = 0
PROCESS_SCALE = 1.0
PREPROCESS_USE_PINNED = True
PRED_MASK_DTYPE = np.uint8

# Export
SAVE_RESULTS = True
RESULTS_DIR = '/mmsegmentation/output/results_clockwork_adaptive_1a_kc30ks100'
SUMMARY_CSV = os.path.join(RESULTS_DIR, 'summary_gpu_only_global.csv')
DETAILS_CSV = os.path.join(RESULTS_DIR, 'details_gpu_only_per_frame.csv')


In [ ]:

# ============================================================
# UTILITIES
# ============================================================
try:
    from mmseg.structures.dual_task_seg_data_sample import DualTaskSegDataSample as _TestDataSample
except Exception:
    from mmseg.structures import SegDataSample as _TestDataSample


def _label_from_LabelData(lbl):
    import numpy as _np
    if lbl is None:
        return None, None

    x = lbl
    scores = None
    for _ in range(8):
        if x is None:
            return None, scores
        if torch.is_tensor(x):
            return int(x.reshape(-1)[0].detach().cpu().item()), scores
        if isinstance(x, _np.ndarray):
            return int(x.reshape(-1)[0]), scores
        if isinstance(x, (int, float, bool, _np.integer, _np.floating)):
            return int(x), scores
        if isinstance(x, (list, tuple)):
            if len(x) == 0:
                return None, scores
            x = x[0]
            continue

        if hasattr(x, 'item') and callable(getattr(x, 'item')):
            try:
                return int(x.item()), scores
            except Exception:
                pass

        if hasattr(x, 'scores'):
            try:
                s = x.scores
                if torch.is_tensor(s):
                    scores = s.detach().cpu().numpy()
                elif isinstance(s, _np.ndarray):
                    scores = s
                elif isinstance(s, (list, tuple)):
                    scores = _np.asarray(s)
            except Exception:
                pass

        next_x = None
        for k in ('label', 'pred_label', 'data', 'value'):
            if hasattr(x, k):
                next_x = getattr(x, k)
                break
        if next_x is x:
            break
        x = next_x

    return None, scores


def _trigger_from_cls_idx(cls_idx):
    if cls_idx is None:
        return None
    if int(cls_idx) == int(CLS_IDX_STRAIGHT):
        return TRIGGER_IDX_STRAIGHT
    if int(cls_idx) in (int(CLS_IDX_LEFT), int(CLS_IDX_RIGHT)):
        return TRIGGER_IDX_CURVE
    return None


def _trigger_from_sample(sample, cls_idx=None):
    if hasattr(sample, 'pred_trigger_label'):
        idx, scores = _label_from_LabelData(sample.pred_trigger_label)
        if idx is not None:
            return idx, scores

    if hasattr(sample, 'pred_trigger'):
        idx, scores = _label_from_LabelData(sample.pred_trigger)
        if idx is not None:
            return idx, scores

    if cls_idx is None and hasattr(sample, 'pred_label') and sample.pred_label is not None:
        cls_idx, cls_scores = _label_from_LabelData(sample.pred_label)
    else:
        cls_scores = None

    idx = _trigger_from_cls_idx(cls_idx)
    if idx is not None:
        return idx, cls_scores

    return None, cls_scores


class ContextPathClock:
    def __init__(self, bise: torch.nn.Module):
        assert hasattr(bise, 'context_path') and hasattr(bise, 'spatial_path'),                 'El backbone no parece ser BiSeNetV1 con context_path/spatial_path.'
        self.context_path = bise.context_path
        self.cache = None
        self.hold = False
        self._orig_forward = self.context_path.forward
        self._install()

    def _install(self):
        @torch.inference_mode()
        def wrapped_forward(x):
            if self.hold and (self.cache is not None):
                return self.cache
            out = self._orig_forward(x)
            if isinstance(out, (list, tuple)):
                self.cache = tuple(o.detach() if torch.is_tensor(o) else o for o in out)
            elif torch.is_tensor(out):
                self.cache = (out.detach(),)
            else:
                self.cache = out
            return out
        self.context_path.forward = wrapped_forward

    def set_hold(self, flag: bool):
        self.hold = bool(flag)

    def invalidate(self):
        self.cache = None
        self.hold = False


class AdaptiveClockScheduler:
    def __init__(self, k_straight=100, k_curve=30, default_mode='curve'):
        self.k_straight = int(k_straight)
        self.k_curve = int(k_curve)
        self.active_k = self.k_curve if str(default_mode).lower() == 'curve' else self.k_straight
        self.next_fire_idx = 0
        self.last_trigger_idx = None

    def should_fire(self, frame_idx: int) -> bool:
        return int(frame_idx) >= int(self.next_fire_idx)

    def _k_from_trigger(self, trigger_idx):
        if trigger_idx == TRIGGER_IDX_CURVE:
            return self.k_curve
        if trigger_idx == TRIGGER_IDX_STRAIGHT:
            return self.k_straight
        return self.active_k

    def update_after_inference(self, frame_idx: int, trigger_idx, did_fire: bool):
        desired_k = self._k_from_trigger(trigger_idx)
        mode_changed = (desired_k != self.active_k)

        self.last_trigger_idx = trigger_idx
        self.active_k = desired_k

        # Requested policy:
        # if the mode changes STRAIGHT <-> CURVE, the next frame is forced as FIRE.
        if mode_changed:
            self.next_fire_idx = int(frame_idx) + 1
            return

        # If the mode did not change and this frame was FIRE, the next FIRE is scheduled according to the active K.
        if did_fire:
            self.next_fire_idx = int(frame_idx) + int(self.active_k)


class NDArrayInferencer:
    def __init__(self, model):
        self.model = model
        self.model.eval()
        self.device = next(self.model.parameters()).device

        cfg_dp = model.cfg.model.get('data_preprocessor', {})
        size = cfg_dp.get('size', (512, 512))
        self.input_w = int(size[0])
        self.input_h = int(size[1])
        self.bgr_to_rgb = bool(cfg_dp.get('bgr_to_rgb', True))
        mean = np.array(cfg_dp.get('mean', [123.675, 116.28, 103.53]), dtype=np.float32)
        std  = np.array(cfg_dp.get('std',  [58.395, 57.12, 57.375]), dtype=np.float32)

        self.mean = torch.tensor(mean, device=self.device, dtype=torch.float32).view(1, 3, 1, 1)
        self.std  = torch.tensor(std,  device=self.device, dtype=torch.float32).view(1, 3, 1, 1)

        self._meta = dict(
            ori_shape=(self.input_h, self.input_w),
            img_shape=(self.input_h, self.input_w),
            pad_shape=(self.input_h, self.input_w),
            batch_input_shape=(self.input_h, self.input_w),
            scale_factor=(1.0, 1.0),
            padding_size=[0, 0, 0, 0],
        )

        self.use_cuda = (self.device.type == 'cuda')
        self.use_pinned = bool(PREPROCESS_USE_PINNED and self.use_cuda)

        self._resize_hwc = np.empty((self.input_h, self.input_w, 3), dtype=np.uint8)
        self._cpu_hwc_t = None
        self._cpu_hwc_np = None
        self._gpu_hwc_u8 = None
        self._gpu_chw_f32 = None

        if self.use_pinned:
            self._cpu_hwc_t = torch.empty((1, self.input_h, self.input_w, 3), dtype=torch.uint8, pin_memory=True)
            self._cpu_hwc_np = self._cpu_hwc_t[0].numpy()
            self._gpu_hwc_u8 = torch.empty((1, self.input_h, self.input_w, 3), dtype=torch.uint8, device=self.device)
            self._gpu_chw_f32 = torch.empty((1, 3, self.input_h, self.input_w), dtype=torch.float32, device=self.device)

    def _make_data_sample(self):
        ds = _TestDataSample()
        ds.set_metainfo(self._meta.copy())
        return ds

    def _preprocess(self, img_bgr_nd):
        if (img_bgr_nd.shape[1], img_bgr_nd.shape[0]) != (self.input_w, self.input_h):
            cv2.resize(img_bgr_nd, (self.input_w, self.input_h), dst=self._resize_hwc, interpolation=cv2.INTER_LINEAR)
            img = self._resize_hwc
        else:
            img = img_bgr_nd

        if self.use_pinned:
            np.copyto(self._cpu_hwc_np, img)
            self._gpu_hwc_u8.copy_(self._cpu_hwc_t, non_blocking=True)
            chw_u8 = self._gpu_hwc_u8.permute(0, 3, 1, 2)
            if self.bgr_to_rgb:
                chw_u8 = chw_u8[:, [2, 1, 0], :, :]
            self._gpu_chw_f32.copy_(chw_u8)
            self._gpu_chw_f32.sub_(self.mean).div_(self.std)
            return self._gpu_chw_f32

        if self.bgr_to_rgb:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = np.ascontiguousarray(img.transpose(2, 0, 1))
        tensor = torch.from_numpy(img).unsqueeze(0)
        if self.use_cuda:
            tensor = tensor.to(self.device, non_blocking=True)
        tensor = tensor.float()
        tensor = (tensor - self.mean) / self.std
        return tensor

    @torch.inference_mode()
    def predict_only(self, img_bgr_nd):
        inputs = self._preprocess(img_bgr_nd)
        data_samples = [self._make_data_sample()]
        preds = self.model.predict(inputs, data_samples)
        return preds[0]


def open_video_frames(video_path, max_frames=None):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f'No se pudo abrir el video: {video_path}')
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
    limit = total if (max_frames is None) else min(total if total > 0 else max_frames, max_frames)
    idx = 0
    pbar = tqdm(total=(limit if limit else None), desc='Frames', unit='frame')
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if max_frames is not None and idx >= max_frames:
                break
            yield idx, frame
            idx += 1
            pbar.update(1)
    finally:
        pbar.close()
        cap.release()


def maybe_scale_frame(frame):
    if PROCESS_SCALE == 1.0:
        return frame
    return cv2.resize(frame, None, fx=PROCESS_SCALE, fy=PROCESS_SCALE, interpolation=cv2.INTER_AREA)


def build_adaptive_pipeline():
    register_all_modules()
    model = init_model(CONFIG_PATH, CKPT_PATH, device=DEVICE)
    model.eval()

    if not (hasattr(model, 'backbone') and hasattr(model.backbone, 'context_path')):
        raise RuntimeError('El modelo no expone backbone.context_path; no se puede aplicar clockwork por features.')

    return {
        'model': model,
        'ctx_clock': ContextPathClock(model.backbone),
        'inferencer': NDArrayInferencer(model),
    }


def reset_adaptive_state(pipe):
    pipe['ctx_clock'].invalidate()
    return AdaptiveClockScheduler(k_straight=K_STRAIGHT, k_curve=K_CURVE, default_mode='curve')


def warmup_adaptive(pipe, max_frames=60):
    scheduler = reset_adaptive_state(pipe)
    for idx, frame in open_video_frames(INPUT_VIDEO, max_frames=max_frames):
        do_fire = scheduler.should_fire(idx)
        pipe['ctx_clock'].set_hold(not do_fire)
        sample = pipe['inferencer'].predict_only(maybe_scale_frame(frame))

        cls_idx = None
        if hasattr(sample, 'pred_label') and sample.pred_label is not None:
            cls_idx, _ = _label_from_LabelData(sample.pred_label)

        trigger_idx, _ = _trigger_from_sample(sample, cls_idx=cls_idx)
        scheduler.update_after_inference(idx, trigger_idx=trigger_idx, did_fire=do_fire)

        if torch.cuda.is_available():
            torch.cuda.synchronize()


def _summary_global(records):
    vals = np.asarray([r['latency_ms'] for r in records], dtype=np.float64)
    if vals.size == 0:
        raise RuntimeError('No se registraron muestras GPU-only.')
    n = int(vals.size)
    fires = int(sum(r['phase'] == 'FIRE' for r in records))
    holds = int(sum(r['phase'] == 'HOLD' for r in records))
    straight = int(sum(r.get('trigger_name') == 'STRAIGHT' for r in records))
    curve = int(sum(r.get('trigger_name') == 'CURVE' for r in records))
    row = {
        'method': 'clockwork_adaptive',
        'metric': 'GPU_ONLY_GLOBAL',
        'n_timed_frames': n,
        'mean_ms': float(vals.mean()),
        'median_ms': float(np.median(vals)),
        'p95_ms': float(np.percentile(vals, 95)),
        'min_ms': float(vals.min()),
        'max_ms': float(vals.max()),
        'fps_from_mean': float(1000.0 / vals.mean()),
        'fires': fires,
        'holds': holds,
        'fire_ratio': float(fires / n),
        'hold_ratio': float(holds / n),
        'trigger_straight_frames': straight,
        'trigger_curve_frames': curve,
        'k_straight': int(K_STRAIGHT),
        'k_curve': int(K_CURVE),
        'video_path': INPUT_VIDEO,
        'config_path': CONFIG_PATH,
        'ckpt_path': CKPT_PATH,
    }
    return pd.DataFrame([row])


def benchmark_adaptive_gpu_only(pipe, max_frames=None, sample_every=1, sample_offset=0):
    if not torch.cuda.is_available():
        raise RuntimeError('GPU-only requiere CUDA disponible.')

    scheduler = reset_adaptive_state(pipe)
    records = []
    frame_count = 0

    inferencer = pipe['inferencer']
    model = pipe['model']

    for idx, frame in open_video_frames(INPUT_VIDEO, max_frames=max_frames):
        do_fire = scheduler.should_fire(idx)
        pipe['ctx_clock'].set_hold(not do_fire)
        small = maybe_scale_frame(frame)

        take_sample = ((frame_count + sample_offset) % sample_every) == 0

        # Comparable with the realistic notebook:
        # GPU-only measures only model.predict(...), not preprocessing.
        inputs = inferencer._preprocess(small)
        data_samples = [inferencer._make_data_sample()]

        torch.cuda.synchronize()
        ev0 = torch.cuda.Event(enable_timing=True)
        ev1 = torch.cuda.Event(enable_timing=True)
        ev0.record()
        preds = model.predict(inputs, data_samples)
        ev1.record()
        torch.cuda.synchronize()
        latency_ms = float(ev0.elapsed_time(ev1))

        sample = preds[0]

        pred = sample.pred_sem_seg.data
        if pred.ndim == 3 and pred.shape[0] == 1:
            pred = pred[0]
        pred = pred.to(dtype=getattr(torch, str(np.dtype(PRED_MASK_DTYPE).name)))
        pred_mask = pred.detach().cpu().numpy()

        cls_idx = None
        if hasattr(sample, 'pred_label') and sample.pred_label is not None:
            cls_idx, _ = _label_from_LabelData(sample.pred_label)

        trigger_idx, _ = _trigger_from_sample(sample, cls_idx=cls_idx)
        trigger_name = None
        if trigger_idx is not None and 0 <= int(trigger_idx) < len(TRIGGER_NAMES):
            trigger_name = TRIGGER_NAMES[int(trigger_idx)]

        active_k_before = int(scheduler.active_k)
        scheduler.update_after_inference(idx, trigger_idx=trigger_idx, did_fire=do_fire)
        mode_changed = (int(scheduler.active_k) != active_k_before)

        if take_sample:
            records.append({
                'frame_idx': int(idx),
                'phase': 'FIRE' if do_fire else 'HOLD',
                'latency_ms': latency_ms,
                'cls_idx': cls_idx if cls_idx is not None else np.nan,
                'trigger_idx': trigger_idx if trigger_idx is not None else np.nan,
                'trigger_name': trigger_name if trigger_name is not None else 'UNKNOWN',
                'mode_changed': bool(mode_changed),
                'active_k_before': int(active_k_before),
                'active_k_after': int(scheduler.active_k),
                'next_fire_idx_after': int(scheduler.next_fire_idx),
                'pred_shape_h': int(pred_mask.shape[0]),
                'pred_shape_w': int(pred_mask.shape[1]),
            })
        frame_count += 1

    return records, _summary_global(records)



In [ ]:

# ============================================================
# MODEL LOADING
# ============================================================
pipe = build_adaptive_pipeline()
print('✅ Modelo ADAPTATIVO cargado.')
print(f'K_STRAIGHT = {K_STRAIGHT}')
print(f'K_CURVE    = {K_CURVE}')


In [ ]:

# ============================================================
# WARM-UP
# ============================================================
print('Warm-up ADAPTATIVO...')
warmup_adaptive(pipe, max_frames=WARMUP_FRAMES)
print('✅ Warm-up listo.')


In [ ]:

# ============================================================
# GPU-ONLY BENCHMARK (NATURAL GLOBAL AVERAGE FIRE+HOLD)
# ============================================================
records, summary_df = benchmark_adaptive_gpu_only(
    pipe,
    max_frames=MAX_FRAMES,
    sample_every=GPU_SAMPLE_EVERY,
    sample_offset=GPU_SAMPLE_OFFSET,
)

details_df = pd.DataFrame(records)

print('=== RESUMEN GLOBAL GPU-ONLY | CLOCKWORK ADAPTATIVO ===')
display(summary_df)

if SAVE_RESULTS:
    os.makedirs(RESULTS_DIR, exist_ok=True)
    summary_df.to_csv(SUMMARY_CSV, index=False)
    details_df.to_csv(DETAILS_CSV, index=False)
    print('CSV resumen :', SUMMARY_CSV)
    print('CSV detalle :', DETAILS_CSV)
